In [1]:
%load_ext autoreload
%autoreload 2

import sys
import time
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.schema import DEQPermitExtraction
from src.scraper import ensure_project_dirs, sync_deq_permits

load_dotenv(PROJECT_ROOT / ".env")
client = genai.Client()
dirs = ensure_project_dirs(PROJECT_ROOT)

# Automatically creates folders, scrapes DEQ, filters to Northern Region,
# keeps only the latest permit per Registration No., and downloads missing PDFs!
permits_df = sync_deq_permits(PROJECT_ROOT, region_filter="Northern", only_latest_per_facility=True)
permits_df.head(10)


Fetching Virginia DEQ permit table...
Filtered out 10 older/superseded permits for existing Registration Numbers.
Sync complete! 157 downloaded, 0 already on disk. Total active permits: 157


,site_name,reg_app_no,base_reg_no,issue_date,issue_date_dt,program_type,locality,region,pdf_url,local_filename,local_filepath,is_latest_for_facility
5,"Equinix, LLC - CU 1-4",41064-8,41064,08/10/2018,2018-08-10,Article 6 - mNSR,Culpeper Co.,Northern,https://www.deq.virginia.gov/home/showpublishe...,"41064-8 - Equinix, LLC - CU 1-4.pdf",/home/droid/environmental-policy/nova-deq-perm...,True
27,Vienna Cloud,71804-4,71804,11/09/2016,2016-11-09,Article 6 - mNSR,Fairfax Co.,Northern,https://www.deq.virginia.gov/home/showpublishe...,71804-4 - Vienna Cloud.pdf,/home/droid/environmental-policy/nova-deq-perm...,True
28,Coresite Real Estate 12100 Sunrise Valley Drive,72296-5,72296,01/31/2013,2013-01-31,Article 6 - mNSR,Fairfax Co.,Northern,https://www.deq.virginia.gov/home/showpublishe...,72296-5 - Coresite Real Estate 12100 Sunrise V...,/home/droid/environmental-policy/nova-deq-perm...,True
29,"Verizon Business Network Services, Inc.",72367-4,72367,02/26/2020,2020-02-26,Article 6 - mNSR,Loudoun Co.,Northern,https://www.deq.virginia.gov/home/showpublishe...,"72367-4 - Verizon Business Network Services, I...",/home/droid/environmental-policy/nova-deq-perm...,True
30,"Oath, Inc.",72368-4,72368,08/28/2009,2009-08-28,Article 6 - mNSR,Prince William Co.,Northern,https://www.deq.virginia.gov/home/showpublishe...,"72368-4 - Oath, Inc..pdf",/home/droid/environmental-policy/nova-deq-perm...,True
31,Comcast Cablevision of Virginia Incorporated,72374-4,72374,03/27/2014,2014-03-27,Article 6 - mNSR,Prince William Co.,Northern,https://www.deq.virginia.gov/home/showpublishe...,72374-4 - Comcast Cablevision of Virginia Inco...,/home/droid/environmental-policy/nova-deq-perm...,True
32,"Chantilly Technology Partners, LLC/H5 VA",72375-9,72375,03/10/2026,2026-03-10,Article 6 - mNSR,Fairfax Co.,Northern,https://www.deq.virginia.gov/home/showpublishe...,"72375-9 - Chantilly Technology Partners, LLC_H...",/home/droid/environmental-policy/nova-deq-perm...,True
33,22810-22860 International Drive LLC,73158-3,73158,09/16/2003,2003-09-16,Article 6 - mNSR,Loudoun Co.,Northern,https://www.deq.virginia.gov/home/showpublishe...,73158-3 - 22810-22860 International Drive LLC.pdf,/home/droid/environmental-policy/nova-deq-perm...,True
34,21110 Ridgetop Circle LLC,73160-6,73160,10/30/2019,2019-10-30,Article 6 - mNSR,Loudoun Co.,Northern,https://www.deq.virginia.gov/home/showpublishe...,73160-6 - 21110 Ridgetop Circle LLC.pdf,/home/droid/environmental-policy/nova-deq-perm...,True
35,"Digital Realty - Lemur Properties, LLC (VA3)",73162-3,73162,03/05/2010,2010-03-05,Article 6 - mNSR,Fairfax Co.,Northern,https://www.deq.virginia.gov/home/showpublishe...,"73162-3 - Digital Realty - Lemur Properties, L...",/home/droid/environmental-policy/nova-deq-perm...,True


In [7]:
PROMPT = """
You are an environmental regulatory data engineer extracting structured data from a Virginia DEQ air permit.
1. Extract the facility name, primary single street address (for geocoding), any additional campus addresses, registration number, fuel specifications, and facility-wide combined annual emission limits (tons/yr).
2. Check if the permit states it combines terms from or supersedes a prior permit document.
3. Extract every diesel engine-generator group from the Equipment List (both 'Equipment to be Constructed/Modified' and 'Previously Permitted Equipment') and Emission Limits tables.
4. Set equipment_status appropriately so active Previously Permitted units are preserved and any Removed/Retired units are flagged.
5. Check carefully for Selective Catalytic Reduction (SCR) or Tier 4 / Tier 4-equivalent limits.
6. For unit ranges like 'EG-1 through EG-15', set unit_prefix='EG-', start_unit_number=1, end_unit_number=15.
"""

total = len(permits_df)

# Test on the first 3 permits first! Remove .head(3) when ready for all 157.
subset = permits_df
total = len(subset)
for idx, (_, row) in enumerate(subset.iterrows(), start=1):

#for idx, (_, row) in enumerate(permits_df.iterrows(), start=1):
    pdf_path: Path = row["local_filepath"]
    reg_app_no = sanitize_id = row["reg_app_no"]
    cache_file = dirs["json_cache"] / f"{reg_app_no}.json"

    # 1. Skip if already extracted in our JSON cache!
    if cache_file.exists():
        print(f"[{idx}/{total}] Skipping {reg_app_no} (already cached in json_cache/)")
        continue

    if not pdf_path.exists():
        print(f"[{idx}/{total}] Missing PDF file for {reg_app_no}, skipping...")
        continue

    print(f"[{idx}/{total}] Extracting {pdf_path.name} via Gemini...")
    uploaded_pdf = None
    try:
        
        with open(pdf_path, "rb") as f:
            uploaded_pdf = client.files.upload(
                file=f,
                config=types.UploadFileConfig(
                    mime_type="application/pdf",
                    display_name=f"{reg_app_no}.pdf" # Pure ASCII name (e.g., '74121-2.pdf')!
                ),
            )

        
        response = None
        for attempt in range(1, 4):
            try:
                response = client.models.generate_content(
                    model="gemini-3.8-flash",
                    contents=[uploaded_pdf, PROMPT],
                    config=types.GenerateContentConfig(
                        response_mime_type="application/json",
                        response_schema=DEQPermitExtraction,
                        temperature=0.0,
                    ),
                )
                break  # Success! Exit retry loop
            except Exception as api_err:
                if "429" in str(api_err) or "RESOURCE_EXHAUSTED" in str(api_err):
                    wait_time = 30 * attempt
                    print(f"    [Rate limit hit] Waiting {wait_time}s before retry {attempt}/3...")
                    time.sleep(wait_time)
                else:
                    raise api_err

        # Validate JSON with Pydantic and save immediately to checkpoint cache
        permit_obj = DEQPermitExtraction.model_validate_json(response.text)
        cache_file.write_text(permit_obj.model_dump_json(indent=2), encoding="utf-8")
        print(f"  -> Saved {cache_file.name} ({len(permit_obj.generator_groups)} groups)")

    except Exception as e:
        print(f"  -> ERROR on {reg_app_no}: {e}")

    finally:
        # Clean up temporary file from Google's servers
        if uploaded_pdf:
            try:
                client.files.delete(name=uploaded_pdf.name)
            except Exception:
                pass

    # Polite 6-second pause to stay safely under Free Tier RPM limits
    time.sleep(6.0)

print("\nBatch extraction complete!")


[1/157] Skipping 41064-8 (already cached in json_cache/)
[2/157] Skipping 71804-4 (already cached in json_cache/)
[3/157] Skipping 72296-5 (already cached in json_cache/)
[4/157] Skipping 72367-4 (already cached in json_cache/)
[5/157] Skipping 72368-4 (already cached in json_cache/)
[6/157] Skipping 72374-4 (already cached in json_cache/)
[7/157] Skipping 72375-9 (already cached in json_cache/)
[8/157] Skipping 73158-3 (already cached in json_cache/)
[9/157] Skipping 73160-6 (already cached in json_cache/)
[10/157] Skipping 73162-3 (already cached in json_cache/)
[11/157] Skipping 73170-1 (already cached in json_cache/)
[12/157] Skipping 73180-4 (already cached in json_cache/)
[13/157] Skipping 73199-3 (already cached in json_cache/)
[14/157] Skipping 73200-3 (already cached in json_cache/)
[15/157] Skipping 73230-2 (already cached in json_cache/)
[16/157] Skipping 73233-8 (already cached in json_cache/)
[17/157] Skipping 73240-4 (already cached in json_cache/)
[18/157] Skipping 73293

In [8]:
all_generator_rows = []
json_files = sorted(dirs["json_cache"].glob("*.json"))

for jf in json_files:
    permit_obj = DEQPermitExtraction.model_validate_json(jf.read_text(encoding="utf-8"))
    rows = permit_obj.to_individual_generator_rows()
    for r in rows:
        r["permit_file_id"] = jf.stem  # e.g., '73670-14'
    all_generator_rows.extend(rows)

master_df = pd.DataFrame(all_generator_rows)
master_csv_path = dirs["processed"] / "all_nova_generators.csv"
master_df.to_csv(master_csv_path, index=False)

print(f"Compiled {len(json_files)} permits into {len(master_df)} individual active generators!")
print(f"Saved master dataset to: {master_csv_path}")

# Show any rows flagged by our 100-ton NOx or PM2.5 > NOx sanity checks:
flagged = master_df[master_df["audit_flags"] != "PASS"]
print(f"Rows with audit flags to review: {len(flagged)}")
master_df.head(15)


Compiled 157 permits into 12707 individual active generators!
Saved master dataset to: /home/droid/environmental-policy/nova-deq-permits/data/processed/all_nova_generators.csv
Rows with audit flags to review: 139


,registration_number,facility_name,primary_street_address,additional_campus_addresses,county_or_city,permit_issue_date,supersedes_prior_permit,superseded_permit_dates,fuel_type,max_fuel_sulfur_ppm,...,has_scr_control,max_operating_hours_per_unit_yr,nox_lb_hr_per_unit,co_lb_hr_per_unit,pm25_lb_hr_per_unit,voc_lb_hr_per_unit,so2_lb_hr_per_unit,source_page_numbers,verbatim_evidence_quote,permit_file_id
0,41064,Equinix,18155 Technology Drive,,Culpeper County,2018-08-10,True,"January 13, 2016 (as amended March 9, 2018)",Ultra-Low Sulfur Diesel (ULSD),15.0,...,True,500.0,4.81,6.09,NaN,1.12,NaN,"5, 9, 11",Three (3) Caterpillar 3516C-HD Diesel Engine G...,41064-8
1,41064,Equinix,18155 Technology Drive,,Culpeper County,2018-08-10,True,"January 13, 2016 (as amended March 9, 2018)",Ultra-Low Sulfur Diesel (ULSD),15.0,...,True,500.0,4.81,6.09,NaN,1.12,NaN,"5, 9, 11",Three (3) Caterpillar 3516C-HD Diesel Engine G...,41064-8
2,41064,Equinix,18155 Technology Drive,,Culpeper County,2018-08-10,True,"January 13, 2016 (as amended March 9, 2018)",Ultra-Low Sulfur Diesel (ULSD),15.0,...,True,500.0,4.81,6.09,NaN,1.12,NaN,"5, 9, 11",Three (3) Caterpillar 3516C-HD Diesel Engine G...,41064-8
3,41064,Equinix,18155 Technology Drive,,Culpeper County,2018-08-10,True,"January 13, 2016 (as amended March 9, 2018)",Ultra-Low Sulfur Diesel (ULSD),15.0,...,False,NaN,46.36,9.33,NaN,0.98,NaN,"5, 10",Six (6) Caterpillar 3516B TA Diesel Engine Gen...,41064-8
4,41064,Equinix,18155 Technology Drive,,Culpeper County,2018-08-10,True,"January 13, 2016 (as amended March 9, 2018)",Ultra-Low Sulfur Diesel (ULSD),15.0,...,False,NaN,46.36,9.33,NaN,0.98,NaN,"5, 10",Six (6) Caterpillar 3516B TA Diesel Engine Gen...,41064-8
5,41064,Equinix,18155 Technology Drive,,Culpeper County,2018-08-10,True,"January 13, 2016 (as amended March 9, 2018)",Ultra-Low Sulfur Diesel (ULSD),15.0,...,False,NaN,46.36,9.33,NaN,0.98,NaN,"5, 10",Six (6) Caterpillar 3516B TA Diesel Engine Gen...,41064-8
6,41064,Equinix,18155 Technology Drive,,Culpeper County,2018-08-10,True,"January 13, 2016 (as amended March 9, 2018)",Ultra-Low Sulfur Diesel (ULSD),15.0,...,False,NaN,46.36,9.33,NaN,0.98,NaN,"5, 10",Six (6) Caterpillar 3516B TA Diesel Engine Gen...,41064-8
7,41064,Equinix,18155 Technology Drive,,Culpeper County,2018-08-10,True,"January 13, 2016 (as amended March 9, 2018)",Ultra-Low Sulfur Diesel (ULSD),15.0,...,False,NaN,46.36,9.33,NaN,0.98,NaN,"5, 10",Six (6) Caterpillar 3516B TA Diesel Engine Gen...,41064-8
8,41064,Equinix,18155 Technology Drive,,Culpeper County,2018-08-10,True,"January 13, 2016 (as amended March 9, 2018)",Ultra-Low Sulfur Diesel (ULSD),15.0,...,False,NaN,46.36,9.33,NaN,0.98,NaN,"5, 10",Six (6) Caterpillar 3516B TA Diesel Engine Gen...,41064-8
9,41064,Equinix,18155 Technology Drive,,Culpeper County,2018-08-10,True,"January 13, 2016 (as amended March 9, 2018)",Ultra-Low Sulfur Diesel (ULSD),15.0,...,True,NaN,4.81,5.86,NaN,1.17,NaN,"5, 10",Six (6) Caterpillar 3516C DITA Diesel Engine G...,41064-8


In [9]:
# Cell 4: Quick Dataset Health & Policy Summary
print(f"Unique Facilities Extracted: {master_df['registration_number'].nunique()} / 157")
print(f"Total Active Generators Unrolled: {len(master_df):,}")

print("\n--- Generators by County/City ---")
print(master_df["county_or_city"].value_counts(dropna=False))

print("\n--- Generators by EPA Tier & SCR Control ---")
print(master_df.groupby(["epa_tier_level", "has_scr_control"]).size())

print("\n--- Facilities with Audit Flags ---")
flagged_facilities = master_df[master_df["audit_flags"] != "PASS"][
    ["permit_file_id", "facility_name", "facility_nox_tpy_limit", "facility_pm25_tpy_limit", "audit_flags"]
].drop_duplicates()
display(flagged_facilities)


Unique Facilities Extracted: 155 / 157
Total Active Generators Unrolled: 12,707

--- Generators by County/City ---
county_or_city
Loudoun County                                4563
Loudoun                                       2328
Prince William County                         1846
Caroline County                               1332
Prince William                                 740
Fairfax County                                 532
Spotsylvania County                            397
Prince William County and City of Manassas     266
Louisa County                                  180
Culpeper County                                157
Stafford County                                 98
City of Manassas                                81
Fairfax                                         48
Fauquier County                                 35
Culpeper                                        35
Fauquier                                        31
Manassas                                        20
Res

,permit_file_id,facility_name,facility_nox_tpy_limit,facility_pm25_tpy_limit,audit_flags
337,73670-14,Digital Loudoun Parkway Center N LLC,167.1,2.0,HIGH_NOX_TPY (167.1)
